# ZIP-ZICO simulation study: 30 independent datasets

This notebook applies the original ZIP-ZICO fitting routine to 30 independently generated datasets.

For seed $s=1,\ldots,30$, it expects:

- `X_seed_s.tsv`: the ZIP count-data matrix
- `B_true_seed_s.tsv`: the corresponding true adjacency matrix, stored as `B_true[parent, child] = 1`

For each dataset, the notebook fits ZIP-ZICO once, evaluates `W0`, `W1`, and `|W0| + |W1|`, saves the seed-level outputs, and summarizes uncertainty across the 30 independent datasets. The ZICO model itself is not modified.

## 1. Disable the incompatible CUDA device before importing PyTorch

In [26]:
import os

# The installed PyTorch build does not support the RTX 4070 sm_89 architecture.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

print("CUDA disabled for this notebook.")

CUDA disabled for this notebook.


## 2. Check lightweight dependencies

In [27]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "torch": "torch",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}

missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Installation completed. Restart the kernel and run all cells again.")
else:
    print("All required packages are available.")

All required packages are available.


## 3. Imports and ZICO path

In [ ]:
from pathlib import Path
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

PROJECT_DIR = Path("ZIO")
if not (PROJECT_DIR / "code" / "helper").is_dir():
    if (Path("code") / "helper").is_dir():
        PROJECT_DIR = Path(".")
    else:
        raise FileNotFoundError("Run Jupyter from ZIO/ or the folder containing ZIO/.")
PROJECT_DIR = PROJECT_DIR.resolve()
ZICO_DIR = PROJECT_DIR / "code" / "simulation" / "4.1" / "ZICO-main"

if not ZICO_DIR.exists():
    raise FileNotFoundError(f"ZICO folder not found: {ZICO_DIR}")

if str(ZICO_DIR) not in sys.path:
    sys.path.insert(0, str(ZICO_DIR))

from zico import ZICO

print("Python:", sys.executable)
print("PyTorch version:", torch.__version__)
print("CUDA visible to PyTorch:", torch.cuda.is_available())
print("ZICO loaded from:", ZICO_DIR)

Python: c:\Users\ftask\AppData\Local\Programs\Python\Python313\python.exe
PyTorch version: 2.13.0+cpu
CUDA visible to PyTorch: False
ZICO loaded from: E:\Study\UT-Dallas\Woody\Project 2\ZICO-main


## 4. Study settings

In [29]:
DATA_DIR = PROJECT_DIR / "data" / "simulation_data_75"


RESULT_DIR = PROJECT_DIR / "output" / "simulation" / "4.1" / "zico_30_independent_datasets_75"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = list(range(1, 31))
X_PATTERN = "X_seed_{seed}.tsv"
B_PATTERN = "B_true_seed_{seed}.tsv"

DEVICE = "cpu"
LOSS_TYPE = "Poisson"
EPOCHS = 3000
LAM = 1e-3
LAMBDA_ALIGN = 1.0
THRESHOLD = 0.3
SAVE_MATRICES = True

print("Current directory:", Path.cwd())
print("Data directory:", DATA_DIR)
print("Result directory:", RESULT_DIR)
print("Seeds:", SEEDS[0], "through", SEEDS[-1])
print("Device:", DEVICE)

Current directory: e:\Study\UT-Dallas\Woody\Project 2\ZICO-main
Data directory: e:\Study\UT-Dallas\Woody\Project 2\ZICO-main\ZICO_simulation_data_75
Result directory: e:\Study\UT-Dallas\Woody\Project 2\ZICO-main\zico_30_independent_datasets_75
Seeds: 1 through 30
Device: cpu


## 5. Confirm all input files exist

In [30]:
missing_files = []

for seed in SEEDS:
    x_file = DATA_DIR / X_PATTERN.format(seed=seed)
    b_file = DATA_DIR / B_PATTERN.format(seed=seed)

    if not x_file.exists():
        missing_files.append(str(x_file))
    if not b_file.exists():
        missing_files.append(str(b_file))

if missing_files:
    print("Missing files:")
    for path in missing_files:
        print("  ", path)
    raise FileNotFoundError(f"{len(missing_files)} required files are missing.")

print("All 30 data files and all 30 reference files were found.")

All 30 data files and all 30 reference files were found.


## 6. Helper functions

In [31]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def evaluate_adjacency(true_B, est_B, threshold=0.3, eps=1e-8):
    true_tensor = torch.as_tensor(true_B).detach().cpu().float()
    est_tensor = torch.as_tensor(est_B).detach().cpu().float()

    if true_tensor.shape != est_tensor.shape:
        raise ValueError(
            f"Shape mismatch: true={tuple(true_tensor.shape)}, "
            f"estimated={tuple(est_tensor.shape)}"
        )

    d = true_tensor.shape[0]
    off_diagonal = ~torch.eye(d, dtype=torch.bool)

    true_binary = (true_tensor != 0).float()
    est_binary = (est_tensor.abs() > threshold).float()

    true_vector = true_binary[off_diagonal]
    est_vector = est_binary[off_diagonal]
    score_vector = est_tensor.abs()[off_diagonal]

    tp = (est_vector * true_vector).sum()
    fp = (est_vector * (1.0 - true_vector)).sum()
    fn = ((1.0 - est_vector) * true_vector).sum()

    tpr = tp / (tp + fn + eps)
    fdr = fp / (tp + fp + eps)

    y_true = true_vector.numpy().astype(int)
    y_score = score_vector.numpy()
    auprc = average_precision_score(y_true, y_score)

    return {
        "TP": int(tp.item()),
        "FP": int(fp.item()),
        "FN": int(fn.item()),
        "TPR": float(tpr.item()),
        "FDR": float(fdr.item()),
        "AUPRC": float(auprc),
        "EstimatedEdges": int((tp + fp).item()),
        "TrueEdges": int((tp + fn).item()),
    }


def validate_dataset(X_np, B_true_np, seed):
    if X_np.ndim != 2:
        raise ValueError(f"Seed {seed}: X must be two-dimensional.")

    n, d = X_np.shape

    if B_true_np.shape != (d, d):
        raise ValueError(
            f"Seed {seed}: B_true shape is {B_true_np.shape}; expected {(d, d)}."
        )

    if np.any(X_np < 0):
        raise ValueError(f"Seed {seed}: X contains negative values.")

    if not np.allclose(X_np, np.round(X_np)):
        raise ValueError(f"Seed {seed}: X contains non-integer values.")

    if not np.allclose(np.diag(B_true_np), 0):
        raise ValueError(f"Seed {seed}: B_true diagonal is not zero.")

    return n, d

## 7. Fit ZIP-ZICO for seeds 1–30

In [32]:
all_metrics = []
run_status = []
study_start = time.perf_counter()

for run_number, seed in enumerate(SEEDS, start=1):
    print("\n" + "=" * 70)
    print(f"Seed {seed} ({run_number}/{len(SEEDS)})")
    print("=" * 70)

    seed_start = time.perf_counter()

    try:
        set_all_seeds(seed)

        x_file = DATA_DIR / X_PATTERN.format(seed=seed)
        b_file = DATA_DIR / B_PATTERN.format(seed=seed)

        X_df = pd.read_csv(x_file, sep="\t")
        B_true_df = pd.read_csv(b_file, sep="\t")

        X_np = X_df.to_numpy(dtype=np.float32)
        B_true_np = B_true_df.to_numpy(dtype=np.float32)

        n, d = validate_dataset(X_np, B_true_np, seed)

        X = torch.from_numpy(X_np).to(DEVICE)
        B_true = torch.from_numpy(B_true_np).to(DEVICE)

        print(
            f"n={n}, p={d}, true edges={int((B_true_np != 0).sum())}, "
            f"zero rate={(X_np == 0).mean():.4f}"
        )

        # Original ZIP-ZICO fitting call.
        model = ZICO(d, device=DEVICE).to(DEVICE)
        model.fit_logdet_batch(
            X,
            max_iter=EPOCHS,
            loss_type=LOSS_TYPE,
            lam=LAM,
            lambda_align=LAMBDA_ALIGN,
        )

        W0 = model.W0.detach().cpu()
        W1 = model.W1.detach().cpu()
        W_combined = W0.abs() + W1.abs()

        seed_results = {
            "W0": evaluate_adjacency(B_true.cpu(), W0, threshold=THRESHOLD),
            "W1": evaluate_adjacency(B_true.cpu(), W1, threshold=THRESHOLD),
            "|W0| + |W1|": evaluate_adjacency(
                B_true.cpu(), W_combined, threshold=THRESHOLD
            ),
        }

        elapsed = time.perf_counter() - seed_start

        for estimate, result in seed_results.items():
            all_metrics.append({
                "Seed": seed,
                "Estimate": estimate,
                **result,
                "ZeroRate": float((X_np == 0).mean()),
                "RuntimeSeconds": elapsed,
                "Threshold": THRESHOLD,
                "Lambda": LAM,
                "LambdaAlign": LAMBDA_ALIGN,
                "Epochs": EPOCHS,
            })

        seed_dir = RESULT_DIR / f"seed_{seed:02d}"
        seed_dir.mkdir(parents=True, exist_ok=True)

        if SAVE_MATRICES:
            np.save(seed_dir / "W0.npy", W0.numpy())
            np.save(seed_dir / "W1.npy", W1.numpy())
            np.save(seed_dir / "W_combined.npy", W_combined.numpy())
            np.save(seed_dir / "B_true.npy", B_true_np)

        pd.DataFrame([
            {"Estimate": name, **values}
            for name, values in seed_results.items()
        ]).to_csv(seed_dir / "metrics.csv", index=False)

        run_status.append({
            "Seed": seed,
            "Status": "completed",
            "RuntimeSeconds": elapsed,
            "Error": "",
        })

        # Checkpoint after every completed seed.
        pd.DataFrame(all_metrics).to_csv(
            RESULT_DIR / "all_seed_metrics_in_progress.csv", index=False
        )
        pd.DataFrame(run_status).to_csv(
            RESULT_DIR / "run_status.csv", index=False
        )

        print(f"Completed seed {seed} in {elapsed:.2f} seconds.")
        display(pd.DataFrame(seed_results).T[
            ["TP", "FP", "FN", "TPR", "FDR", "AUPRC"]
        ].round(4))

    except Exception as exc:
        elapsed = time.perf_counter() - seed_start

        run_status.append({
            "Seed": seed,
            "Status": "failed",
            "RuntimeSeconds": elapsed,
            "Error": repr(exc),
        })

        pd.DataFrame(run_status).to_csv(
            RESULT_DIR / "run_status.csv", index=False
        )

        print(f"Seed {seed} failed: {exc}")

metrics_df = pd.DataFrame(all_metrics)
status_df = pd.DataFrame(run_status)

metrics_df.to_csv(RESULT_DIR / "all_seed_metrics.csv", index=False)
status_df.to_csv(RESULT_DIR / "run_status.csv", index=False)

total_runtime = time.perf_counter() - study_start

print("\n" + "=" * 70)
print("Batch run finished.")
print("Completed seeds:", (status_df["Status"] == "completed").sum())
print("Failed seeds:", (status_df["Status"] == "failed").sum())
print(f"Total runtime: {total_runtime:.2f} seconds")
print("Results:", RESULT_DIR)


Seed 1 (1/30)
n=1000, p=75, true edges=75, zero rate=0.4884


100%|██████████| 3000/3000 [00:37<00:00, 80.47it/s, grp=163, h=0.00249, mu=0.001, nll=103]


Completed seed 1 in 37.35 seconds.


,TP,FP,FN,TPR,FDR,AUPRC
W0,63.0,7.0,12.0,0.84,0.1000,0.8009
W1,63.0,7.0,12.0,0.84,0.1000,0.8008
|W0| + |W1|,66.0,24.0,9.0,0.88,0.2667,0.8008



Seed 2 (2/30)
n=1000, p=75, true edges=75, zero rate=0.4883


100%|██████████| 3000/3000 [00:36<00:00, 82.79it/s, grp=167, h=0.00214, mu=0.001, nll=105]


... (111 outputs left out)

100%|██████████| 3000/3000 [00:35<00:00, 85.21it/s, grp=177, h=0.00169, mu=0.001, nll=101]


Completed seed 30 in 35.29 seconds.


,TP,FP,FN,TPR,FDR,AUPRC
W0,58.0,9.0,17.0,0.7733,0.1343,0.7798
W1,58.0,9.0,17.0,0.7733,0.1343,0.7798
|W0| + |W1|,64.0,45.0,11.0,0.8533,0.4128,0.7798



Batch run finished.
Completed seeds: 30
Failed seeds: 0
Total runtime: 1167.49 seconds
Results: e:\Study\UT-Dallas\Woody\Project 2\ZICO-main\zico_30_independent_datasets_75


## 8. Review run status

In [33]:
display(status_df)

failed_runs = status_df.loc[status_df["Status"] == "failed"]

if failed_runs.empty:
    print("All seeds completed successfully.")
else:
    print("The following seeds failed:")
    display(failed_runs)

,Seed,Status,RuntimeSeconds,Error
0,1,completed,37.353385,
1,2,completed,36.330098,
2,3,completed,38.086475,
3,4,completed,35.800748,
4,5,completed,37.009052,
5,6,completed,35.497166,
6,7,completed,34.114962,
7,8,completed,37.360189,
8,9,completed,38.362714,
9,10,completed,35.926646,


All seeds completed successfully.


## 9. Seed-level metrics

In [34]:
if metrics_df.empty:
    raise RuntimeError("No seed completed successfully.")

display(metrics_df[[
    "Seed", "Estimate", "TP", "FP", "FN", "TPR", "FDR", "AUPRC",
    "EstimatedEdges", "TrueEdges", "ZeroRate", "RuntimeSeconds"
]].round(4))

,Seed,Estimate,TP,FP,FN,TPR,FDR,AUPRC,EstimatedEdges,TrueEdges,ZeroRate,RuntimeSeconds
0,1,W0,63,7,12,0.8400,0.1000,0.8009,70,75,0.4884,37.3534
1,1,W1,63,7,12,0.8400,0.1000,0.8008,70,75,0.4884,37.3534
2,1,|W0| + |W1|,66,24,9,0.8800,0.2667,0.8008,90,75,0.4884,37.3534
3,2,W0,60,11,15,0.8000,0.1549,0.5961,71,75,0.4883,36.3301
4,2,W1,60,11,15,0.8000,0.1549,0.5961,71,75,0.4883,36.3301
...,...,...,...,...,...,...,...,...,...,...,...,...
85,29,W1,61,12,14,0.8133,0.1644,0.6755,73,75,0.4813,37.8287
86,29,|W0| + |W1|,62,24,13,0.8267,0.2791,0.6755,86,75,0.4813,37.8287
87,30,W0,58,9,17,0.7733,0.1343,0.7798,67,75,0.4935,35.2919
88,30,W1,58,9,17,0.7733,0.1343,0.7798,67,75,0.4935,35.2919


## 10. Uncertainty summary across independent datasets

The standard deviation describes between-dataset variability. The standard error is

$$SE = \frac{SD}{\sqrt{R}},$$

where $R$ is the number of successfully completed independent datasets. The notebook also reports the normal-approximation interval $\bar{x} \pm 1.96SE$.

In [35]:
summary_metrics = [
    "TP", "FP", "FN", "TPR", "FDR", "AUPRC",
    "EstimatedEdges", "ZeroRate", "RuntimeSeconds"
]

summary_rows = []

for estimate, group in metrics_df.groupby("Estimate"):
    for metric in summary_metrics:
        values = group[metric].dropna()
        count = int(values.count())
        mean = float(values.mean())
        sd = float(values.std(ddof=1)) if count > 1 else np.nan
        se = float(values.sem(ddof=1)) if count > 1 else np.nan

        summary_rows.append({
            "Estimate": estimate,
            "Metric": metric,
            "N": count,
            "Mean": mean,
            "SD": sd,
            "SE": se,
            "CI95_Lower": mean - 1.96 * se if count > 1 else np.nan,
            "CI95_Upper": mean + 1.96 * se if count > 1 else np.nan,
        })

summary_long = pd.DataFrame(summary_rows)
summary_long.to_csv(
    RESULT_DIR / "summary_mean_sd_se_ci95_long.csv", index=False
)

display(summary_long.round(4))

,Estimate,Metric,N,Mean,SD,SE,CI95_Lower,CI95_Upper
0,W0,TP,30,60.2333,3.0590,0.5585,59.1387,61.3280
1,W0,FP,30,10.0333,2.9182,0.5328,8.9891,11.0776
2,W0,FN,30,14.7667,3.0590,0.5585,13.6720,15.8613
3,W0,TPR,30,0.8031,0.0408,0.0074,0.7885,0.8177
4,W0,FDR,30,0.1423,0.0391,0.0071,0.1283,0.1563
5,W0,AUPRC,30,0.7262,0.0820,0.0150,0.6969,0.7556
6,W0,EstimatedEdges,30,70.2667,2.8999,0.5294,69.2290,71.3044
7,W0,ZeroRate,30,0.4877,0.0113,0.0021,0.4836,0.4918
8,W0,RuntimeSeconds,30,38.8901,6.2158,1.1349,36.6658,41.1145
9,W1,TP,30,60.2333,3.0590,0.5585,59.1387,61.3280


## 11. Compact mean ± SE table

In [36]:
main_metrics = ["TP", "FP", "FN", "TPR", "FDR", "AUPRC", "EstimatedEdges"]
compact_rows = []

for estimate, group in metrics_df.groupby("Estimate"):
    row = {"Estimate": estimate}
    for metric in main_metrics:
        mean = group[metric].mean()
        se = group[metric].sem(ddof=1)
        row[metric] = f"{mean:.4f} ± {se:.4f}"
    compact_rows.append(row)

compact_summary = pd.DataFrame(compact_rows)
compact_summary.to_csv(
    RESULT_DIR / "summary_mean_plus_minus_se.csv", index=False
)

display(compact_summary)

,Estimate,TP,FP,FN,TPR,FDR,AUPRC,EstimatedEdges
0,W0,60.2333 ± 0.5585,10.0333 ± 0.5328,14.7667 ± 0.5585,0.8031 ± 0.0074,0.1423 ± 0.0071,0.7262 ± 0.0150,70.2667 ± 0.5294
1,W1,60.2333 ± 0.5585,10.0333 ± 0.5328,14.7667 ± 0.5585,0.8031 ± 0.0074,0.1423 ± 0.0071,0.7262 ± 0.0150,70.2667 ± 0.5294
2,|W0| + |W1|,62.8667 ± 0.5768,29.5000 ± 1.4885,12.1333 ± 0.5768,0.8382 ± 0.0077,0.3147 ± 0.0115,0.7262 ± 0.0150,92.3667 ± 1.5331


## 12. Wide mean, SD, and SE table

In [37]:
wide_summary = (
    metrics_df
    .groupby("Estimate")[main_metrics]
    .agg(["mean", "std", "sem"])
)

wide_summary.to_csv(RESULT_DIR / "summary_mean_sd_se_wide.csv")
display(wide_summary.round(4))

TP                       FP                       FN  \
                mean     std     sem     mean     std     sem     mean   
Estimate                                                                 
W0           60.2333  3.0590  0.5585  10.0333  2.9182  0.5328  14.7667   
W1           60.2333  3.0590  0.5585  10.0333  2.9182  0.5328  14.7667   
|W0| + |W1|  62.8667  3.1594  0.5768  29.5000  8.1526  1.4885  12.1333   

                                TPR  ...             FDR                  \
                std     sem    mean  ...     sem    mean     std     sem   
Estimate                             ...                                   
W0           3.0590  0.5585  0.8031  ...  0.0074  0.1423  0.0391  0.0071   
W1           3.0590  0.5585  0.8031  ...  0.0074  0.1423  0.0391  0.0071   
|W0| + |W1|  3.1594  0.5768  0.8382  ...  0.0077  0.3147  0.0629  0.0115   

              AUPRC               EstimatedEdges                  
               mean    std    sem           mean     std     sem  
Estimate                                                          
W0           0.7262  0.082  0.015        70.2667  2.8999  0.5294  
W1           0.7262  0.082  0.015        70.2667  2.8999  0.5294  
|W0| + |W1|  0.7262  0.082  0.015        92.3667  8.3974  1.5331  

[3 rows x 21 columns]

## 13. Per-seed comparisons

In [38]:
for metric in ["TPR", "FDR", "AUPRC"]:
    comparison = metrics_df.pivot(
        index="Seed", columns="Estimate", values=metric
    )

    print(f"\n{metric} by seed")
    display(comparison.round(4))
    comparison.to_csv(RESULT_DIR / f"{metric.lower()}_by_seed.csv")


TPR by seed


Estimate,W0,W1,|W0| + |W1|
Seed,,,
1,0.8400,0.8400,0.8800
2,0.8000,0.8000,0.8267
3,0.8400,0.8400,0.8667
4,0.7733,0.7733,0.8133
5,0.7867,0.7867,0.8000
6,0.8667,0.8667,0.9200
7,0.8400,0.8400,0.8800
8,0.7200,0.7200,0.7467
9,0.8533,0.8533,0.8800



FDR by seed


Estimate,W0,W1,|W0| + |W1|
Seed,,,
1,0.1000,0.1000,0.2667
2,0.1549,0.1549,0.2955
3,0.1127,0.1127,0.3299
4,0.1212,0.1212,0.2738
5,0.1806,0.1806,0.2941
6,0.1447,0.1447,0.3301
7,0.1127,0.1127,0.2500
8,0.1692,0.1692,0.3636
9,0.1233,0.1233,0.2826



AUPRC by seed


Estimate,W0,W1,|W0| + |W1|
Seed,,,
1,0.8009,0.8008,0.8008
2,0.5961,0.5961,0.5961
3,0.8353,0.8353,0.8353
4,0.6600,0.6601,0.6600
5,0.6407,0.6407,0.6407
6,0.8092,0.8091,0.8091
7,0.8069,0.8069,0.8069
8,0.6218,0.6217,0.6217
9,0.7592,0.7591,0.7591


## Main saved outputs

- `all_seed_metrics.csv`: all seed-level performance results
- `run_status.csv`: completed and failed seeds
- `summary_mean_plus_minus_se.csv`: compact mean ± SE summary
- `summary_mean_sd_se_ci95_long.csv`: mean, SD, SE, and 95% interval
- `seed_01/` through `seed_30/`: estimated matrices and seed-specific metrics

Because progress is checkpointed after every seed, completed results remain available if a later seed fails or the notebook is interrupted.